In [7]:
import re
from pathlib import Path
from typing import List, Set, Dict
from loguru import logger

In [8]:
RAW_DATA_DIR = Path("../data/raw")

In [9]:
SYLLABUS_SCHEMA: Set[str] = {
    "General information",
    "Materials",
    "Learning outcomes",
    "CLOs-PLOs maps",
    "Sessions",
    "Constructive questions",
    "Assignments and assessments"
}

CURRICULUM_SCHEMA: Set[str] = {
    "General information",
    "Programme outcomes",
    "Program learning outcomes",
    "PLOs-POs maps",
    "Curriculum details"
}

PATHWAY_SCHEMA: Set[str] = {
    "General information"
}

In [10]:
RE_H1_SYLLABUS = re.compile(r"^# Syllabus:", re.MULTILINE)
RE_H1_CURRICULUM = re.compile(r"^# Curriculum:", re.MULTILINE)
RE_H1_PATHWAY = re.compile(r"^# Pathway:", re.MULTILINE)
RE_H2 = re.compile(r"^##\s+(.*)", re.MULTILINE)

In [11]:
def extract_h2_headers(content: str) -> Set[str]:
    """Extracts all Level 2 Markdown headers from a given string."""
    headers = RE_H2.findall(content)
    return {h.strip() for h in headers}

In [12]:
def validate_corpus_schemas(data_dir: Path) -> Dict[str, dict]:
    """
    Scans the markdown corpus and validates each file against its schema.

    1. Validate syllabus schema
    2. Validate curriculum schema
    3. Validate pathway schema
        Metrics collected:
            - Total documents per schema
            - Compliance rate per schema
            - List of unrecognized documents
        Logs detailed warnings for non-compliant documents and critical errors for encoding issues.

    Args:
        data_dir (Path): Path to the directory containing markdown files.

    Returns: a dictionary summarizing the validation results.
    """
    logger.info(f"Initiating Schema Validation pipeline for corpus at: {data_dir}")

    md_files: List[Path] = list(data_dir.glob("*.[mM][dD]"))
    if not md_files:
        logger.error("No markdown files found. Terminating validation.")
        return {}

    logger.info(f"Discovered {len(md_files)} files for validation.")

    metrics = {
        "syllabus": {
            "total": 0,
            "compliant": 0
        },
        "curriculum": {
            "total": 0,
            "compliant": 0
        },
        "pathway": {
            "total": 0,
            "compliant": 0
        },
        "unrecognized": []
    }

    for file_path in md_files:
        try:
            with open(file_path, "r", encoding="utf-8", errors="strict") as f:
                content = f.read()

            h2_set = extract_h2_headers(content)

            if RE_H1_SYLLABUS.search(content):
                metrics["syllabus"]["total"] += 1
                if SYLLABUS_SCHEMA.issubset(h2_set):
                    metrics["syllabus"]["compliant"] += 1
                else:
                    missing = SYLLABUS_SCHEMA - h2_set
                    logger.warning(f"[Syllabus Violation] {file_path.name} missing: {missing}")

            elif RE_H1_CURRICULUM.search(content):
                metrics["curriculum"]["total"] += 1
                if CURRICULUM_SCHEMA.issubset(h2_set):
                    metrics["curriculum"]["compliant"] += 1
                else:
                    missing = CURRICULUM_SCHEMA - h2_set
                    logger.warning(f"[Curriculum Violation] {file_path.name} missing: {missing}")

            elif RE_H1_PATHWAY.search(content):
                metrics["pathway"]["total"] += 1
                if PATHWAY_SCHEMA.issubset(h2_set):
                    metrics["pathway"]["compliant"] += 1
                else:
                    missing = PATHWAY_SCHEMA - h2_set
                    logger.warning(f"[Pathway Violation] {file_path.name} missing: {missing}")

            else:
                metrics["unrecognized"].append(file_path.name)
                logger.error(f"[Unknown Schema] {file_path.name} lacks valid H1 Entity ID.")

        except UnicodeDecodeError:
            logger.critical(f"Encoding failure on {file_path.name}. Must be UTF-8.")

    return metrics

In [13]:
def log_validation_report(metrics: Dict[str, dict]) -> None:
    """Computes compliance rates and logs the final data engineering report."""
    logger.info("Schema Validation Report")

    for category in ["syllabus", "curriculum", "pathway"]:
        total = metrics[category]["total"]
        if total > 0:
            rate = (metrics[category]["compliant"] / total) * 100
            logger.info(f"{category.capitalize()} Documents: {total}")
            if rate == 100.0:
                logger.success(f"{category.capitalize()} Compliance: {rate:.2f}%")
            else:
                logger.warning(f"{category.capitalize()} Compliance: {rate:.2f}%")

    if metrics["unrecognized"]:
        logger.error(f"Failed to classify {len(metrics['unrecognized'])} documents.")
        for doc in metrics["unrecognized"]:
            logger.error(f"   -> {doc}")

In [14]:
validation_metrics = validate_corpus_schemas(RAW_DATA_DIR)

2026-03-10 01:02:44.512 | INFO     | __main__:validate_corpus_schemas:19 - Initiating Schema Validation pipeline for corpus at: ..\data\raw
2026-03-10 01:02:44.514 | INFO     | __main__:validate_corpus_schemas:26 - Discovered 64 files for validation.


In [15]:
if validation_metrics:
    log_validation_report(validation_metrics)

2026-03-10 01:02:44.576 | INFO     | __main__:log_validation_report:3 - Schema Validation Report
2026-03-10 01:02:44.576 | INFO     | __main__:log_validation_report:9 - Syllabus Documents: 56
2026-03-10 01:02:44.576 | SUCCESS  | __main__:log_validation_report:11 - Syllabus Compliance: 100.00%
2026-03-10 01:02:44.576 | INFO     | __main__:log_validation_report:9 - Curriculum Documents: 3
2026-03-10 01:02:44.576 | SUCCESS  | __main__:log_validation_report:11 - Curriculum Compliance: 100.00%
2026-03-10 01:02:44.576 | INFO     | __main__:log_validation_report:9 - Pathway Documents: 5
2026-03-10 01:02:44.576 | SUCCESS  | __main__:log_validation_report:11 - Pathway Compliance: 100.00%
